<a href="https://colab.research.google.com/github/Mitul-Marimuthu/deep-learning/blob/phase2/cat_dog_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Switch runtime to T4 GPU To ensure GPU availability.

In [2]:
!pip install torch torchvision matplotlib numpy -q

import torch
import torchvision
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.10.0+cu128
GPU available: True


In [4]:
# Download directly from Microsoft's URL (original source)
!wget -q https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q kagglecatsanddogs_5340.zip -d cats_dogs
!ls cats_dogs/PetImages

Cat  Dog


In [6]:
# Set up the dataset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

# some images are corrupted --> remove them
import os
from PIL import Image

def remove_corrupted(folder):
    removed = 0
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        try:
            img = Image.open(fpath)
            img.verify()
        except:
            os.remove(fpath)
            removed += 1
    print(f"Removed {removed} corrupted images from {folder}")

remove_corrupted('cats_dogs/PetImages/Cat')
remove_corrupted('cats_dogs/PetImages/Dog')

# Define transforms - resize everything to 224 x 224, normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
# specific mean/std values are ImageNet dataset statistics.
# use them because resnet was pretained on ImageNet
# normalizing them with the same values it was trained on
# makes transfer learning work much better

# Load dataset
# Automatically assigns labels based on folder names.
# Cat -> 0, Dog -> 1
full_dataset = datasets.ImageFolder('cats_dogs/PetImages', transform=transform)
print(f"Total images: {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}") # ['Cat', 'Dog']

# 80/20 train/val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


Removed 0 corrupted images from cats_dogs/PetImages/Cat
Removed 0 corrupted images from cats_dogs/PetImages/Dog
Total images: 24998
Classes: ['Cat', 'Dog']
Train batches: 625, Val batches: 157
